# temperature와 샘플링

같은 프롬프트에서 temperature를 바꾸면 반복 출력의 다양성이 어떻게 달라질까요?

Python 3.12와 잠긴 패키지를 사용합니다. 첫 실행에서는 로컬 모델을 다운로드합니다.

## 로컬 생성 모델 준비

In [1]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, logging, set_seed

logging.set_verbosity_error()

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
REVISION = "7ae557604adf67be50417f59c2c2f167def9a775"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION)
model.eval()
print("model:", MODEL_ID)
print("parameters:", f"{model.num_parameters():,}")

model: Qwen/Qwen2.5-0.5B-Instruct
parameters: 494,032,768


In [2]:
def generate(prompt: str, *, temperature: float = 0.7, seed: int = 0, max_new_tokens: int = 48) -> str:
    # 채팅 템플릿을 적용한 뒤 새로 생성된 토큰만 복원합니다.
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt")
    set_seed(seed)
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=temperature if temperature > 0 else None,
            top_p=0.9 if temperature > 0 else None,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## temperature별 반복 생성

프롬프트, top-p, 최대 생성 길이를 고정하고 temperature와 난수 시드를 바꾸어 실제 출력을 모두 보존합니다.

In [3]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
prompt = "오프라인 서점의 가을 행사 이름을 한국어 5어절 이내로 하나 제안하세요."
records = []
for temperature in [0.3, 1.0]:
    for seed in [11, 22, 33]:
        answer = generate(prompt, temperature=temperature, seed=seed, max_new_tokens=24)
        records.append((temperature, seed, answer))
        print(f"T={temperature:.1f} seed={seed}: {answer}")

T=0.3 seed=11: "열정과 풍부한 책: 오프라인 서점의 가을 행사를 맞이


T=0.3 seed=22: 1. "열매와 함께: 가을의 풍경"
2. "오늘의 날


T=0.3 seed=33: "가을의 푸른 시선: 오프라인 서점의 봄행사"


T=1.0 seed=11: 죄송합니다, 저는 실시간 정보나 실제 위치에 맞는 인적 자원을 가지고 있지 않습니다. 하지만


T=1.0 seed=22: 1. "열린 시선" (Sunlit Eye)
2. "월드를 뚫고"


T=1.0 seed=33: "온라인 어워서행 특별 판매"


## 서로 다른 출력 수

문자열이 정확히 같은지만 세어, 표현의 다양성을 제한적으로 관찰합니다.

In [4]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
for temperature in [0.3, 1.0]:
    answers = [answer for t, _, answer in records if t == temperature]
    print(f"T={temperature:.1f} unique={len(set(answers))}/3")
print("문자열 종류 수는 제안의 품질이나 창의성을 직접 측정하지 않습니다.")

T=0.3 unique=3/3
T=1.0 unique=3/3
문자열 종류 수는 제안의 품질이나 창의성을 직접 측정하지 않습니다.


## 구조 검증

예상한 실험 조건과 출력 구조가 유지되었는지 확인합니다.

In [5]:
assert len(records) == 6
assert {t for t, _, _ in records} == {0.3, 1.0}
print("검증: 2개 temperature × 동일한 3개 시드의 출력을 확인했습니다.")

검증: 2개 temperature × 동일한 3개 시드의 출력을 확인했습니다.
